# Preprocessing data

In [6]:
import pandas as pd

### Keterangan Dataset

In [7]:
# Load dataset hasil scraping
df = pd.read_csv('dataset_detik.csv')

# 1. Cek Informasi Awal
print("Informasi Dataset Awal")
print(f"Total Baris Data : {len(df)} baris")
print(f"Daftar Fitur Awal : {list(df.columns)}")

# Buat kolom id jika belum tersedia pada dataset hasil scraping
if 'id' not in df.columns:
    df.insert(0, 'id', range(1, len(df) + 1))

# 2. Filter Fitur yang Digunakan
df = df[['id', 'isi_berita', 'label']].copy()
print(f"\nDaftar Fitur Digunakan : {list(df.columns)}")
print("\nKeterangan Fitur yang Digunakan:")
print("- id         : identitas unik untuk setiap berita")
print("- isi_berita : teks atau isi lengkap berita")
print("- label      : kategori berita, yaitu sport atau finance")

# 3. Tampilkan baris data
print("\nScraping Selesai! Data disimpan ke 'dataset_detik.csv'")
df.head(5)

Informasi Dataset Awal
Total Baris Data : 200 baris
Daftar Fitur Awal : ['id', 'isi_berita', 'label', 'url']

Daftar Fitur Digunakan : ['id', 'isi_berita', 'label']

Keterangan Fitur yang Digunakan:
- id         : identitas unik untuk setiap berita
- isi_berita : teks atau isi lengkap berita
- label      : kategori berita, yaitu sport atau finance

Scraping Selesai! Data disimpan ke 'dataset_detik.csv'


,id,isi_berita,label
0,1,Ketua Umum Komite Olahraga Nasional Indonesia ...,sport
1,2,Pengurus Pusat Kickboxing Indonesia (PP KBI) p...,sport
2,3,"Lina Hisage, atlet tolak peluru asal Wamena, P...",sport
3,4,Timnas Equestrian Indonesia sudah bertolak ke ...,sport
4,5,"Dalam dua seri terakhir MotoGP 2027, Marc Marq...",sport


### Inisialisasi Library Sastrawi

In [3]:
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi Sastrawi
stop_factory = StopWordRemoverFactory()
stopword_remover = stop_factory.create_stop_word_remover()

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

### Case Folding

In [19]:
# 1. Eksekusi Case Folding (mengubah huruf kapital menjadi huruf kecil)
df['step1_case_folding'] = df['isi_berita'].str.lower()

# 2. Tampilkan Perbandingan Sebelum dan Sesudah (Tabel Pandas)
print("=== PERBANDINGAN SEBELUM & SESUDAH CASE FOLDING (TABEL) ===")
display(df[['id', 'isi_berita', 'step1_case_folding', 'label']].head(3))

# 3. Tampilkan Detail Teks 1 Sampel Artikel untuk Membandingkan Karakter
print("\n=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===")
print("🔴 SEBELUM (isi_berita):")
print(df['isi_berita'].iloc[0][:200] + "...\n")

print("🟢 SESUDAH (step1_case_folding):")
print(df['step1_case_folding'].iloc[0][:200] + "...")

=== PERBANDINGAN SEBELUM & SESUDAH CASE FOLDING (TABEL) ===


,id,isi_berita,step1_case_folding,label
0,1,Ketua Umum Komite Olahraga Nasional Indonesia ...,ketua umum komite olahraga nasional indonesia ...,sport
1,2,Pengurus Pusat Kickboxing Indonesia (PP KBI) p...,pengurus pusat kickboxing indonesia (pp kbi) p...,sport
2,3,"Lina Hisage, atlet tolak peluru asal Wamena, P...","lina hisage, atlet tolak peluru asal wamena, p...",sport



=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===
🔴 SEBELUM (isi_berita):
Ketua Umum Komite Olahraga Nasional Indonesia (KONI) Pusat Letjen TNI Purn Marciano Norman resmi menutup program Sport Diplomacy antara Indonesia dan Timor Leste. Kegiatan ini berlangsung pada 11-14 S...

🟢 SESUDAH (step1_case_folding):
ketua umum komite olahraga nasional indonesia (koni) pusat letjen tni purn marciano norman resmi menutup program sport diplomacy antara indonesia dan timor leste. kegiatan ini berlangsung pada 11-14 s...


### Cleaning

In [ ]:
import re

In [16]:
# 1. Definisi fungsi cleaning
def clean_text(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Hapus URL/link
    text = re.sub(r'\d+', '', text)                   # Hapus angka
    text = re.sub(r'[^\w\s]', ' ', text)              # Hapus tanda baca & simbol khusus
    text = re.sub(r'\s+', ' ', text).strip()          # Hapus spasi berlebih & spasi di awal/akhir
    return text

# 2. Terapkan cleaning pada hasil Case Folding (step1_case_folding)
df['step2_cleaning'] = df['step1_case_folding'].apply(clean_text)

# 3. Tampilkan Perbandingan Sebelum dan Sesudah Cleaning (Tabel Pandas)
print("=== PERBANDINGAN SEBELUM & SESUDAH CLEANING (TABEL) ===")
display(df[['id', 'step1_case_folding', 'step2_cleaning', 'label']].head())

# 4. Tampilkan Detail Teks 1 Sampel Artikel (ID: 1)
print("\nDETAIL CONTOH TEKS BARIS PERTAMA")
print("🔴 SEBELUM CLEANING (step1_case_folding):")
print(df['step1_case_folding'].iloc[0][:200] + "...\n")

print("🟢 SESUDAH CLEANING (step2_cleaning):")
print(df['step2_cleaning'].iloc[0][:200] + "...")

=== PERBANDINGAN SEBELUM & SESUDAH CLEANING (TABEL) ===


,id,step1_case_folding,step2_cleaning,label
0,1,ketua umum komite olahraga nasional indonesia ...,ketua umum komite olahraga nasional indonesia ...,sport
1,2,pengurus pusat kickboxing indonesia (pp kbi) p...,pengurus pusat kickboxing indonesia pp kbi per...,sport
2,3,"lina hisage, atlet tolak peluru asal wamena, p...",lina hisage atlet tolak peluru asal wamena pap...,sport
3,4,timnas equestrian indonesia sudah bertolak ke ...,timnas equestrian indonesia sudah bertolak ke ...,sport
4,5,"dalam dua seri terakhir motogp 2027, marc marq...",dalam dua seri terakhir motogp marc marquez su...,sport



=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===
🔴 SEBELUM CLEANING (step1_case_folding):
ketua umum komite olahraga nasional indonesia (koni) pusat letjen tni purn marciano norman resmi menutup program sport diplomacy antara indonesia dan timor leste. kegiatan ini berlangsung pada 11-14 s...

🟢 SESUDAH CLEANING (step2_cleaning):
ketua umum komite olahraga nasional indonesia koni pusat letjen tni purn marciano norman resmi menutup program sport diplomacy antara indonesia dan timor leste kegiatan ini berlangsung pada september ...


### Tokenization

In [22]:
# 1. Terapkan Tokenization pada hasil Cleaning (step2_cleaning)
# Menggunakan .split() karena teks sudah bersih dari tanda baca
df['step3_tokenize'] = df['step2_cleaning'].apply(lambda x: x.split())

# 2. Tampilkan Perbandingan Sebelum dan Sesudah Tokenization (Tabel Pandas)
print("=== PERBANDINGAN SEBELUM & SESUDAH TOKENIZATION (TABEL) ===")
display(df[['id', 'step2_cleaning', 'step3_tokenize', 'label']].head())

# 3. Tampilkan Detail Teks 1 Sampel Artikel (ID: 1)
print("\n=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===")
print("🔴 SEBELUM TOKENIZATION (step2_cleaning - Tipe Data: String):")
print(df['step2_cleaning'].iloc[0][:150] + "...\n")

print("🟢 SESUDAH TOKENIZATION (step3_tokenize - Tipe Data: List of Tokens):")
print(df['step3_tokenize'].iloc[0][:15]) # Tampilkan 15 kata/token pertama

=== PERBANDINGAN SEBELUM & SESUDAH TOKENIZATION (TABEL) ===


,id,step2_cleaning,step3_tokenize,label
0,1,ketua umum komite olahraga nasional indonesia ...,"[ketua, umum, komite, olahraga, nasional, indo...",sport
1,2,pengurus pusat kickboxing indonesia pp kbi per...,"[pengurus, pusat, kickboxing, indonesia, pp, k...",sport
2,3,lina hisage atlet tolak peluru asal wamena pap...,"[lina, hisage, atlet, tolak, peluru, asal, wam...",sport
3,4,timnas equestrian indonesia sudah bertolak ke ...,"[timnas, equestrian, indonesia, sudah, bertola...",sport
4,5,dalam dua seri terakhir motogp marc marquez su...,"[dalam, dua, seri, terakhir, motogp, marc, mar...",sport



=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===
🔴 SEBELUM TOKENIZATION (step2_cleaning - Tipe Data: String):
ketua umum komite olahraga nasional indonesia koni pusat letjen tni purn marciano norman resmi menutup program sport diplomacy antara indonesia dan ti...

🟢 SESUDAH TOKENIZATION (step3_tokenize - Tipe Data: List of Tokens):
['ketua', 'umum', 'komite', 'olahraga', 'nasional', 'indonesia', 'koni', 'pusat', 'letjen', 'tni', 'purn', 'marciano', 'norman', 'resmi', 'menutup']


### Normalisasi Slang & Istilah Asing

In [24]:
# 1. Definisi Kamus Normalisasi (Slang, Singkatan, & Istilah Asing/Olahraga/Keuangan)
kamus_normalisasi = {
    # Singkatan & Slang Umum Bahasa Indonesia
    'yg': 'yang',
    'dgn': 'dengan',
    'utk': 'untuk',
    'dr': 'dari',
    'sbg': 'sebagai',
    'tdk': 'tidak',
    'krn': 'karena',
    'sdh': 'sudah',
    'blm': 'belum',
    
    # Istilah Serapan / Inggris Kategori Sport & Finance
    'match': 'tanding',
    'coaching': 'pelatih',
    'coach': 'pelatih',
    'strikers': 'striker',
    'goal': 'gol',
    'percent': 'persen',
    'company': 'perusahaan',
    'tax': 'pajak'
}

# 2. Fungsi Normalisasi per List Token
def normalize_tokens(tokens):
    return [kamus_normalisasi.get(word, word) for word in tokens]

# 3. Terapkan Normalisasi pada step3_tokenize
df['step4_normalized'] = df['step3_tokenize'].apply(normalize_tokens)

# 4. Tampilkan Perbandingan Sebelum dan Sesudah Normalisasi (Tabel Pandas)
print("=== PERBANDINGAN SEBELUM & SESUDAH NORMALISASI (TABEL) ===")
display(df[['id', 'step3_tokenize', 'step4_normalized', 'label']].head())

# 5. Tampilkan Detail Teks 1 Sampel Artikel (ID: 1)
print("\n=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===")
print("🔴 SEBELUM NORMALISASI (step3_tokenize):")
print(df['step3_tokenize'].iloc[0][:15])

print("\n🟢 SESUDAH NORMALISASI (step4_normalized):")
print(df['step4_normalized'].iloc[0][:15])

=== PERBANDINGAN SEBELUM & SESUDAH NORMALISASI (TABEL) ===


,id,step3_tokenize,step4_normalized,label
0,1,"[ketua, umum, komite, olahraga, nasional, indo...","[ketua, umum, komite, olahraga, nasional, indo...",sport
1,2,"[pengurus, pusat, kickboxing, indonesia, pp, k...","[pengurus, pusat, kickboxing, indonesia, pp, k...",sport
2,3,"[lina, hisage, atlet, tolak, peluru, asal, wam...","[lina, hisage, atlet, tolak, peluru, asal, wam...",sport
3,4,"[timnas, equestrian, indonesia, sudah, bertola...","[timnas, equestrian, indonesia, sudah, bertola...",sport
4,5,"[dalam, dua, seri, terakhir, motogp, marc, mar...","[dalam, dua, seri, terakhir, motogp, marc, mar...",sport



=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===
🔴 SEBELUM NORMALISASI (step3_tokenize):
['ketua', 'umum', 'komite', 'olahraga', 'nasional', 'indonesia', 'koni', 'pusat', 'letjen', 'tni', 'purn', 'marciano', 'norman', 'resmi', 'menutup']

🟢 SESUDAH NORMALISASI (step4_normalized):
['ketua', 'umum', 'komite', 'olahraga', 'nasional', 'indonesia', 'koni', 'pusat', 'letjen', 'tni', 'purn', 'marciano', 'norman', 'resmi', 'menutup']


### Stopword Removal

In [26]:
# 1. Terapkan Stopword Removal pada hasil Normalisasi (step4_normalized)
# List token digabung menjadi string (' '.join) agar dapat diproses oleh Sastrawi stopword_remover
df['step5_stopword'] = df['step4_normalized'].apply(
    lambda tokens: stopword_remover.remove(' '.join(tokens))
)

# 2. Tampilkan Perbandingan Sebelum dan Sesudah Stopword Removal (Tabel Pandas)
print("=== PERBANDINGAN SEBELUM & SESUDAH STOPWORD REMOVAL (TABEL) ===")
display(df[['id', 'step4_normalized', 'step5_stopword', 'label']].head())

# 3. Tampilkan Detail Teks 1 Sampel Artikel (ID: 1)
print("\n=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===")
print("🔴 SEBELUM STOPWORD REMOVAL (step4_normalized):")
print(df['step4_normalized'].iloc[0][:15])

print("\n🟢 SESUDAH STOPWORD REMOVAL (step5_stopword):")
print(df['step5_stopword'].iloc[0][:150] + "...")

=== PERBANDINGAN SEBELUM & SESUDAH STOPWORD REMOVAL (TABEL) ===


,id,step4_normalized,step5_stopword,label
0,1,"[ketua, umum, komite, olahraga, nasional, indo...",ketua umum komite olahraga nasional indonesia ...,sport
1,2,"[pengurus, pusat, kickboxing, indonesia, pp, k...",pengurus pusat kickboxing indonesia pp kbi per...,sport
2,3,"[lina, hisage, atlet, tolak, peluru, asal, wam...",lina hisage atlet tolak peluru asal wamena pap...,sport
3,4,"[timnas, equestrian, indonesia, sudah, bertola...",timnas equestrian indonesia bertolak jepang je...,sport
4,5,"[dalam, dua, seri, terakhir, motogp, marc, mar...",dua seri terakhir motogp marc marquez sukses s...,sport



=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===
🔴 SEBELUM STOPWORD REMOVAL (step4_normalized):
['ketua', 'umum', 'komite', 'olahraga', 'nasional', 'indonesia', 'koni', 'pusat', 'letjen', 'tni', 'purn', 'marciano', 'norman', 'resmi', 'menutup']

🟢 SESUDAH STOPWORD REMOVAL (step5_stopword):
ketua umum komite olahraga nasional indonesia koni pusat letjen tni purn marciano norman resmi menutup program sport diplomacy indonesia timor leste k...


### Stemming

In [28]:
# 1. Terapkan Stemming pada hasil Stopword Removal (step5_stopword)
print("Sedang memproses Stemming (proses ini memakan waktu 1–3 menit)...")
df['step6_stemming'] = df['step5_stopword'].apply(lambda x: stemmer.stem(x))

# Tetapkan kolom hasil bersih akhir
df['isi_berita_clean'] = df['step6_stemming']

# 2. Tampilkan Perbandingan Sebelum dan Sesudah Stemming (Tabel Pandas)
print("\n=== PERBANDINGAN SEBELUM & SESUDAH STEMMING (TABEL) ===")
display(df[['id', 'step5_stopword', 'step6_stemming', 'label']].head(3))

# 3. Tampilkan Detail Teks 1 Sampel Artikel (ID: 1)
print("\n=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===")
print("🔴 SEBELUM STEMMING (step5_stopword):")
print(df['step5_stopword'].iloc[0][:150] + "...\n")

print("🟢 SESUDAH STEMMING (step6_stemming / isi_berita_clean):")
print(df['step6_stemming'].iloc[0][:150] + "...")

Sedang memproses Stemming (proses ini memakan waktu 1–3 menit)...

=== PERBANDINGAN SEBELUM & SESUDAH STEMMING (TABEL) ===


,id,step5_stopword,step6_stemming,label
0,1,ketua umum komite olahraga nasional indonesia ...,ketua umum komite olahraga nasional indonesia ...,sport
1,2,pengurus pusat kickboxing indonesia pp kbi per...,urus pusat kickboxing indonesia pp kbi periode...,sport
2,3,lina hisage atlet tolak peluru asal wamena pap...,lina hisage atlet tolak peluru asal wamena pap...,sport



=== DETAIL CONTOH TEKS BARIS PERTAMA (ID: 1) ===
🔴 SEBELUM STEMMING (step5_stopword):
ketua umum komite olahraga nasional indonesia koni pusat letjen tni purn marciano norman resmi menutup program sport diplomacy indonesia timor leste k...

🟢 SESUDAH STEMMING (step6_stemming / isi_berita_clean):
ketua umum komite olahraga nasional indonesia koni pusat letjen tni purn marciano norman resmi tutup program sport diplomacy indonesia timor leste gia...


In [32]:
# 4. Simpan Hasil Preprocessing Lengkap ke CSV
df_preprocessed = df[['id', 'isi_berita_clean', 'label']].copy()
df_preprocessed.to_csv('detik_preprocessed.csv', index=False, encoding='utf-8-sig')

print("\n✅ Seluruh 6 Tahap Preprocessing Selesai!")
print("💾 Dataset bersih akhir tersimpan sebagai 'detik_preprocessed.csv'")


✅ Seluruh 6 Tahap Preprocessing Selesai!
💾 Dataset bersih akhir tersimpan sebagai 'detik_preprocessed.csv'


In [38]:
# 1. Pastikan DataFrame df_preprocessed atau file detik_preprocessed.csv siap
if 'df_preprocessed' not in locals():
    df_preprocessed = pd.read_csv('detik_preprocessed.csv')

# 2. Ringkasan Informasi Dataset Hasil Preprocessing
print("=== RINGKASAN DATASET PREPROCESSED ===")
print(f"• Total Baris Data : {len(df_preprocessed)} berita")
print(f"• Daftar Fitur     : {list(df_preprocessed.columns)}")
print(f"• Distribusi Label :\n{df_preprocessed['label'].value_counts().to_string()}\n")

# 3. Tampilkan Tabel Preview Data Hasil Preprocessing (5 Baris Pertama)
print("=== TABEL DATASET PREPROCESSED (HEAD) ===")
display(df_preprocessed.head(5))

# 4. Tampilkan Sampel Acak Per Kategori (2 Sport & 2 Finance)
print("\n=== SAMPEL DATA PER LABEL (SPORT & FINANCE) ===")

# Ambil sampel lalu urutkan descending berdasarkan label
sampel_df = df_preprocessed.groupby('label').sample(5, random_state=42)
sampel_df = sampel_df.sort_values(by='label', ascending=False)

display(sampel_df)

=== RINGKASAN DATASET PREPROCESSED ===
• Total Baris Data : 200 berita
• Daftar Fitur     : ['id', 'isi_berita_clean', 'label']
• Distribusi Label :
label
sport      100
finance    100

=== TABEL DATASET PREPROCESSED (HEAD) ===


,id,isi_berita_clean,label
0,1,ketua umum komite olahraga nasional indonesia ...,sport
1,2,urus pusat kickboxing indonesia pp kbi periode...,sport
2,3,lina hisage atlet tolak peluru asal wamena pap...,sport
3,4,timnas equestrian indonesia tolak jepang jelan...,sport
4,5,dua seri akhir motogp marc marquez sukses sapu...,sport



=== SAMPEL DATA PER LABEL (SPORT & FINANCE) ===


,id,isi_berita_clean,label
79,80,audisi umum pb djarum kudus sedang langsung pe...,sport
37,38,kim antonelli rebut podium pertama seri italia...,sport
65,66,balap vr racing team franco morbidelli jalan h...,sport
54,55,juara renang ajar mula jakarta selatan selesai...,sport
15,16,kontingen atlet ajar jakarta barat jakbar ukir...,sport
183,184,kepala eksekutif awas bursa mineral komoditas ...,finance
153,154,pt industri kereta api persero inka buka suara...,finance
170,171,pt bukit asam tbk ptba ungkap besar beban penu...,finance
145,146,presiden prabowo subianto resmi lrt jakarta ha...,finance
144,145,presiden prabowo subianto resmi panjang layan ...,finance
